# Phase-1 Windows One-Run Pipeline

This notebook runs the existing scripts in order for Windows/VSCode Jupyter:
- Core pipeline: `run_phase1_lowmem.py --reuse-existing`
- Certification path: `08 -> 09 -> 10 -> 11 -> 06 -> 07`


## 0) Setup Path and Runner

Open this notebook from the repository's `Phase-1` folder in VSCode Jupyter.
If needed, set `PHASE1_REPO_DIR` manually.


In [ ]:
import os
import sys
import csv
import json
import subprocess
from pathlib import Path

repo_env = os.environ.get('PHASE1_REPO_DIR', '').strip()
if repo_env:
    REPO_DIR = Path(repo_env).expanduser().resolve()
else:
    REPO_DIR = Path.cwd().resolve()

if (REPO_DIR / 'Phase-1').exists():
    PHASE1_DIR = REPO_DIR / 'Phase-1'
elif REPO_DIR.name == 'Phase-1':
    PHASE1_DIR = REPO_DIR
    REPO_DIR = REPO_DIR.parent
else:
    raise FileNotFoundError(
        f'Cannot locate Phase-1 from REPO_DIR={REPO_DIR}. '
        'Set PHASE1_REPO_DIR to repository root.'
    )

PYTHON = sys.executable
os.environ['PYTHONUNBUFFERED'] = '1'

print('Python   :', PYTHON)
print('Repo dir :', REPO_DIR)
print('Phase-1  :', PHASE1_DIR)


def run_py(script_name: str, *args: str):
    cmd = [PYTHON, '-u', str(PHASE1_DIR / script_name), *args]
    print()
    print('$', ' '.join(cmd))
    r = subprocess.run(cmd, cwd=str(PHASE1_DIR), text=True)
    if r.returncode != 0:
        raise RuntimeError(f'{script_name} failed with exit code {r.returncode}')


## 1) Optional Dependency Install


In [ ]:
INSTALL_DEPS = False

if INSTALL_DEPS:
    req = PHASE1_DIR / 'requirements.txt'
    if req.exists():
        cmd = [PYTHON, '-m', 'pip', 'install', '-r', str(req)]
        print('$', ' '.join(cmd))
        subprocess.run(cmd, check=False)
    else:
        print('requirements.txt not found; skipped.')
else:
    print('Dependency install skipped.')


## 2) Fixed Runtime Profile

These match the reliability-first defaults.


In [ ]:
profile = {
    'PHASE1_DEVICE': 'auto',
    'PHASE1_CUDA_DEVICE': '0',
    'PHASE1_USE_GPU': '1',
    'PHASE1_DOMAIN_BATCH_SIZE': '256',
    'PHASE1_MAX_BATCHES': '0',
    'PHASE1_INDEX_MAX_BATCHES': '0',
    'PHASE1_TFIDF_MAX_FEATURES': '3000',
    'PHASE1_BUILD_TFIDF_MATRIX': '0',
    'PHASE1_STORE_DOC_TEXTS': '1',
    'PHASE1_DOC_TEXT_BACKEND': 'sqlite',
    'PHASE1_DOC_TEXT_INSERT_BATCH': '2000',
    'PHASE1_DOC_TEXT_CACHE_LIMIT': '2500',
    'PHASE1_ENABLE_MINHASH': '1',
    'PHASE1_QUERY_CACHE_LIMIT': '1024',
    'PHASE1_NGRAM_CACHE_LIMIT': '2048',
    'PHASE1_SEMANTIC_CANDIDATE_LIMIT': '80',
    'PHASE1_NGRAM_CANDIDATE_LIMIT': '80',
    'PHASE1_SKIP_REDUNDANCY': '0',
    'PHASE1_SKIP_PERPLEXITY': '0',
    'PHASE1_DATASETS_CONFIG': 'datasets_config.json',
    'PHASE1_IDENTITY_CONFIG': 'configs/metric_identity_v1.json',
}

for k, v in profile.items():
    os.environ[k] = str(v)

print('Profile loaded.')


## 3) Run Core Pipeline (Resume-safe)

Runs: `run_phase1_lowmem.py --reuse-existing`


In [ ]:
run_py('run_phase1_lowmem.py', '--reuse-existing')


## 4) Generate Label Templates (08)

If label files already have manual labels, skip this cell to avoid overwrite.


In [ ]:
RUN_08 = True
if RUN_08:
    run_py('08_generate_label_templates.py')
else:
    print('Skipped 08.')


## 5) Label Completeness Check

`09` requires calibration labels in `ood_labels_v1.csv`.


In [ ]:
labels_dir = PHASE1_DIR / 'validation' / 'labels'
checks = {
    'domain_labels_v1.csv': ('gold_primary', 'main_eval'),
    'quality_labels_v1.csv': ('gold_has_examples', 'main_eval'),
    'difficulty_sanity_v1.csv': ('gold_valid', 'main_eval'),
    'ood_labels_v1.csv': ('gold_in_domain', 'calibration'),
}

def non_empty(v):
    return str(v or '').strip() != ''

ok = True
for file_name, (col, split_name) in checks.items():
    path = labels_dir / file_name
    if not path.exists():
        print(f'[MISS] {file_name}')
        ok = False
        continue

    with path.open('r', encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))

    split_rows = [r for r in rows if str(r.get('split', '')).strip() == split_name]
    labeled = [r for r in split_rows if non_empty(r.get(col))]

    print(f'[CHECK] {file_name}: split={split_name}, total={len(split_rows)}, labeled={len(labeled)}')

    if len(split_rows) == 0 or len(labeled) == 0:
        ok = False

if not ok:
    raise SystemExit('Labeling incomplete. Fill labels and rerun this cell.')

print('Labeling check passed.')


## 6) Run Certification Chain

Runs: `09 -> 10 -> 11 -> 06 -> 07 --require-gates`


In [ ]:
run_py('09_calibrate_ood_thresholds.py')
run_py('10_score_metric_gates.py')
run_py('11_certify_phase1.py')
run_py('06_build_dashboard.py')
run_py('07_validate_phase1_outputs.py', '--require-gates', '--write-report', 'outputs/validation/final_validation_report.json')


## 7) Final Output Check


In [ ]:
targets = [
    PHASE1_DIR / 'outputs' / 'run_manifest.json',
    PHASE1_DIR / 'outputs' / 'run_summary.json',
    PHASE1_DIR / 'outputs' / 'dashboard.html',
    PHASE1_DIR / 'outputs' / 'validation' / 'ood_calibration_report.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'gate_scores_main.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'gate_scores_transfer.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'full_validation_report.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'final_validation_report.json',
]

for p in targets:
    print(f"[{'OK' if p.exists() else 'MISS'}] {p}")

manifest_path = PHASE1_DIR / 'outputs' / 'run_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print()
    print('certification_status:', manifest.get('certification_status'))
    print('certification_failed_gates:', manifest.get('certification_failed_gates'))
